<a href="https://colab.research.google.com/github/talmolab/sleap/blob/main/docs/notebooks/Interactive_and_resumable_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Interactive and resumable training

Most of the time, you will be training models through the GUI or using the [`sleap-nn train` CLI](https://nn.sleap.ai/training).

If you'd like to customize the training process, however, you can use sleap-nn's low-level training functionality interactively. This allows you to define scripts that train models according to your own workflow, for example, to **resume training** on an already trained model. Another possible application would be to train a model using **transfer learning**, where a pretrained model can be used to initialize the weights of the new model.

In this notebook we will explore how to set up a training job and train a model for multiple rounds without the GUI or CLI.

## 1. Setup

Run this cell first to install `sleap-nn`. If you get a dependency error in subsequent cells, just click **Runtime** → **Restart runtime** to reload the packages.

Don't forget to set **Runtime** → **Change runtime type** → **GPU** as the accelerator.

In [1]:
!pip install -qqq "sleap-nn[torch-cpu]"

# if you have GPU (in colab, enable GPU runtime)
# !pip install -qqq "sleap-nn[torch-cuda-128]"

zsh:1: command not found: pip


Import SLEAP to make sure it installed correctly and print out some information about the system:

In [2]:
import sleap_nn

sleap_nn.__version__

'0.0.1'

## 2. Setup training data

Here we will download an existing training dataset package. This is an `.slp` file that contains both the labeled poses, as well as the image data for labeled frames.

If running on Google Colab, you'll want to replace this with mounting your Google Drive folder containing your own data, or if running locally, simply change the path to your labels below in `TRAINING_SLP_FILE`.

In [3]:
# !curl -L --output labels.pkg.slp https://www.dropbox.com/s/b990gxjt3d3j3jh/210205.sleap_wt_gold.13pt.pkg.slp?dl=1
!curl -L --output labels.pkg.slp https://storage.googleapis.com/sleap-data/datasets/wt_gold.13pt/tracking_split2/train.pkg.slp
!ls -lah

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  619M  100  619M    0     0  43.7M      0  0:00:14  0:00:14 --:--:-- 47.0M
total 1283000
drwxr-xr-x@ 14 divyasesh  staff   448B Sep 23 17:41 .
drwxr-xr-x@ 15 divyasesh  staff   480B Sep 23 11:43 ..
-rw-r--r--@  1 divyasesh  staff   713K Sep 22 10:30 Analysis_examples.ipynb
-rw-r--r--@  1 divyasesh  staff   462K Sep 22 12:05 Data_structures.ipynb
-rw-r--r--@  1 divyasesh  staff   175K Sep 22 12:07 Interactive_and_realtime_inference.ipynb
-rw-r--r--@  1 divyasesh  staff    66K Sep 23 17:41 Interactive_and_resumable_training.ipynb
-rw-r--r--@  1 divyasesh  staff   159K Sep 23 16:54 Model_evaluation.ipynb
-rw-r--r--@  1 divyasesh  staff   127K Sep 22 12:06 Post_inference_tracking.ipynb
-rw-r--r--@  1 divyasesh  staff   471K Sep 19 19:36 SLEAP_Tutorial_at_Cosyne_2024_Using_exported_data.ipynb
-rw-r--r--@  1 divyasesh  staff    45K 

In [4]:
TRAINING_SLP_FILE = "labels.pkg.slp"

## 3. Setup training job

A sleap-nn `TrainingJobConfig` is a structure that contains all of the hyperparameters needed to train a SLEAP model. This is typically saved out to `initial_config.yaml` and `training_config.yaml` in the model folder so that training runs can be reproduced if needed, as well as to store metadata necessary for inference.

Normally, these are generated interactively by the GUI, or manually by editing an existing YAML file in a text editor. Here, we will define a configuration interactively entirely in Python.

In [5]:
from sleap_nn.config.training_job_config import TrainingJobConfig, verify_training_cfg
from sleap_nn.config.model_config import UNetConfig, CenteredInstanceConfMapsConfig, CenteredInstanceConfig
from sleap_nn.config.data_config import AugmentationConfig, GeometricConfig


# Initialize the default training job configuration.
cfg = TrainingJobConfig()

# Update path to training data we just downloaded.
cfg.data_config.train_labels_path = [TRAINING_SLP_FILE]
cfg.data_config.validation_fraction = 0.1

# These configures the actual neural network and the model type:
cfg.model_config.backbone_config.unet = UNetConfig(
    filters=16,
    output_stride=4
)
centered_head = CenteredInstanceConfig(confmaps=CenteredInstanceConfMapsConfig(anchor_part="thorax", sigma=1.5, output_stride=4))
cfg.model_config.head_configs.centered_instance = centered_head


# Preprocesssing and training parameters.
cfg.data_config.augmentation_config = AugmentationConfig(geometric=GeometricConfig(affine_p=1.0))
cfg.trainer_config.max_epochs = 5  # This is the maximum number of training rounds.
cfg.trainer_config.train_data_loader.batch_size = 4
cfg.trainer_config.val_data_loader.batch_size = 4

# Setup how we want to save the trained model.
cfg.trainer_config.save_ckpt = True
cfg.trainer_config.run_name = "baseline_model.topdown_centered_instance"

# verify config structure
cfg = verify_training_cfg(cfg)
cfg

{'data_config': {'train_labels_path': ['labels.pkg.slp'], 'val_labels_path': None, 'validation_fraction': 0.1, 'test_file_path': None, 'provider': 'LabelsReader', 'user_instances_only': True, 'data_pipeline_fw': 'torch_dataset', 'cache_img_path': None, 'use_existing_imgs': False, 'delete_cache_imgs_after_training': True, 'preprocessing': {'ensure_rgb': False, 'ensure_grayscale': False, 'max_height': None, 'max_width': None, 'scale': 1.0, 'crop_size': None, 'min_crop_size': 100}, 'use_augmentations_train': False, 'augmentation_config': {'intensity': None, 'geometric': {'rotation_min': -15.0, 'rotation_max': 15.0, 'scale_min': 0.9, 'scale_max': 1.1, 'translate_width': 0.0, 'translate_height': 0.0, 'affine_p': 1.0, 'erase_scale_min': 0.0001, 'erase_scale_max': 0.01, 'erase_ratio_min': 1.0, 'erase_ratio_max': 1.0, 'erase_p': 0.0, 'mixup_lambda_min': 0.01, 'mixup_lambda_max': 0.05, 'mixup_p': 0.0}}, 'skeletons': None}, 'model_config': {'init_weights': 'default', 'pretrained_backbone_weights

Existing configs can also be loaded from a `.yaml` file with:

```python
from omegaconf import OmegaConf
cfg = OmegaConf.load("training_config.yaml")
```

## 4. Training
Next we will create a SLEAP `Trainer` from the configuration we just specified. This handles all the nitty gritty mechanics necessary to setup training in the backend.

In [6]:
from sleap_nn.training.model_trainer import ModelTrainer

trainer = ModelTrainer.get_model_trainer_from_config(cfg)

2025-09-23 17:41:56 | INFO | sleap_nn.training.model_trainer:_setup_train_val_labels:216 | Creating train-val split...
2025-09-23 17:41:57 | INFO | sleap_nn.training.model_trainer:_setup_train_val_labels:261 | # Train Labeled frames: 1440
2025-09-23 17:41:57 | INFO | sleap_nn.training.model_trainer:_setup_train_val_labels:262 | # Val Labeled frames: 160
2025-09-23 17:41:57 | INFO | sleap_nn.training.model_trainer:setup_config:512 | Setting up config...


Great, now we're ready to do the first round of training. This is when the model will actually start to improve over time:

In [7]:
trainer.train()

GPU available: True (mps), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
/Users/divyasesh/Desktop/talmolab/sleap-core-docs/.venv/lib/python3.13/site-packages/torch/utils/data/dataloader.py:684: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)
/Users/divyasesh/Desktop/talmolab/sleap-core-docs/.venv/lib/python3.13/site-packages/lightning/pytorch/callbacks/model_checkpoint.py:751: Checkpoint directory /Users/divyasesh/Desktop/talmolab/sleap-core-docs/docs/notebooks/baseline_model.topdown_centered_instance exists and is not empty.

  | Name                     | Type              | Params | Mode 
-----------------------------------------------------------------------
0 | model                    | Model             | 306 K  | train
1 | instance_peaks_inf_layer | FindInstancePeaks | 0      | train
--------------------------------------------------------

2025-09-23 17:41:58 | INFO | sleap_nn.training.model_trainer:train:849 | Setting up for training...
2025-09-23 17:41:58 | INFO | sleap_nn.training.model_trainer:_setup_model_ckpt_dir:575 | Setting up model ckpt dir: `baseline_model.topdown_centered_instance`...
2025-09-23 17:41:58 | INFO | sleap_nn.training.model_trainer:train:868 | Setting up Trainer...
2025-09-23 17:41:58 | INFO | sleap_nn.training.model_trainer:_setup_loggers_callbacks:647 | Setting up callbacks and loggers...
2025-09-23 17:41:58 | INFO | sleap_nn.training.model_trainer:train:897 | Trainer devices: auto
2025-09-23 17:41:58 | INFO | sleap_nn.training.model_trainer:train:950 | Training on 1 device(s)
2025-09-23 17:41:58 | INFO | sleap_nn.training.model_trainer:train:951 | Training on mps:0 accelerator
2025-09-23 17:41:58 | INFO | sleap_nn.training.model_trainer:train:955 | Setting up lightning module for centered_instance model...
2025-09-23 17:41:58 | INFO | sleap_nn.training.model_trainer:train:959 | Backbone model:

Sanity Checking: |          | 0/? [00:00<?, ?it/s]

/Users/divyasesh/Desktop/talmolab/sleap-core-docs/.venv/lib/python3.13/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:433: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=7` in the `DataLoader` to improve performance.
/Users/divyasesh/Desktop/talmolab/sleap-core-docs/.venv/lib/python3.13/site-packages/lightning/pytorch/core/module.py:520: You called `self.log('learning_rate', ..., logger=True)` but have no logger configured. You can enable one by doing `Trainer(logger=ALogger(...))`
/Users/divyasesh/Desktop/talmolab/sleap-core-docs/.venv/lib/python3.13/site-packages/lightning/pytorch/core/module.py:520: You called `self.log('val_loss', ..., logger=True)` but have no logger configured. You can enable one by doing `Trainer(logger=ALogger(...))`
/Users/divyasesh/Desktop/talmolab/sleap-core-docs/.venv/lib/python3.13/site-packages/lightning/pytorch/core/module.py:520

Training: |          | 0/? [00:00<?, ?it/s]

...

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

`Trainer.fit` stopped: `max_epochs=5` reached.


2025-09-23 17:45:42 | INFO | sleap_nn.training.model_trainer:train:1037 | Finished training loop. [3.7 min]


## 5. Continuing training

We can continue training by setting the `resume_ckpt_path` to the previous ckpt with a potentially different number of epochs:

In [8]:
from pathlib import Path

cfg.trainer_config.max_epochs = 10 #(prv_epochs + extra epochs)
cfg.trainer_config.resume_ckpt_path = Path(cfg.trainer_config.ckpt_dir) / f"{cfg.trainer_config.run_name}" / "best.ckpt"

trainer = ModelTrainer.get_model_trainer_from_config(cfg)
trainer.train()


2025-09-23 17:46:43 | INFO | sleap_nn.training.model_trainer:_setup_train_val_labels:216 | Creating train-val split...
2025-09-23 17:46:43 | INFO | sleap_nn.training.model_trainer:_setup_train_val_labels:261 | # Train Labeled frames: 1440
2025-09-23 17:46:43 | INFO | sleap_nn.training.model_trainer:_setup_train_val_labels:262 | # Val Labeled frames: 160
2025-09-23 17:46:43 | INFO | sleap_nn.training.model_trainer:setup_config:512 | Setting up config...
2025-09-23 17:46:43 | INFO | sleap_nn.training.model_trainer:_setup_ckpt_path:380 | Checkpoint path already exists: baseline_model.topdown_centered_instance... adding suffix to prevent overwriting.
2025-09-23 17:46:43 | INFO | sleap_nn.training.model_trainer:train:849 | Setting up for training...
2025-09-23 17:46:43 | INFO | sleap_nn.training.model_trainer:_setup_model_ckpt_dir:575 | Setting up model ckpt dir: `baseline_model.topdown_centered_instance-1`...


GPU available: True (mps), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
/Users/divyasesh/Desktop/talmolab/sleap-core-docs/.venv/lib/python3.13/site-packages/lightning/pytorch/callbacks/model_checkpoint.py:751: Checkpoint directory /Users/divyasesh/Desktop/talmolab/sleap-core-docs/docs/notebooks/baseline_model.topdown_centered_instance-1 exists and is not empty.
Restoring states from the checkpoint path at baseline_model.topdown_centered_instance/best.ckpt
/Users/divyasesh/Desktop/talmolab/sleap-core-docs/.venv/lib/python3.13/site-packages/lightning/pytorch/callbacks/model_checkpoint.py:445: The dirpath has changed from '/Users/divyasesh/Desktop/talmolab/sleap-core-docs/docs/notebooks/baseline_model.topdown_centered_instance' to '/Users/divyasesh/Desktop/talmolab/sleap-core-docs/docs/notebooks/baseline_model.topdown_centered_instance-1', therefore `best_model_score`, `kth_best_model_path`, `kth_value`, `last_model_path` and `best_k_models` won'

2025-09-23 17:46:43 | INFO | sleap_nn.training.model_trainer:train:868 | Setting up Trainer...
2025-09-23 17:46:43 | INFO | sleap_nn.training.model_trainer:_setup_loggers_callbacks:647 | Setting up callbacks and loggers...
2025-09-23 17:46:43 | INFO | sleap_nn.training.model_trainer:train:897 | Trainer devices: auto
2025-09-23 17:46:43 | INFO | sleap_nn.training.model_trainer:train:950 | Training on 1 device(s)
2025-09-23 17:46:43 | INFO | sleap_nn.training.model_trainer:train:951 | Training on mps:0 accelerator
2025-09-23 17:46:43 | INFO | sleap_nn.training.model_trainer:train:955 | Setting up lightning module for centered_instance model...
2025-09-23 17:46:43 | INFO | sleap_nn.training.model_trainer:train:959 | Backbone model: UNet(
  (encoders): ModuleList(
    (0): Encoder(
      (encoder_stack): ModuleList(
        (0): SimpleConvBlock(
          (blocks): Sequential(
            (stack0_enc0_conv0): Conv2d(1, 16, kernel_size=(3, 3), stride=(1, 1), padding=same)
            (stack

Sanity Checking: |          | 0/? [00:00<?, ?it/s]

Training: |          | 0/? [00:00<?, ?it/s]


Detected KeyboardInterrupt, attempting graceful shutdown ...


2025-09-23 17:47:55 | INFO | sleap_nn.training.model_trainer:train:1037 | Finished training loop. [1.2 min]


SystemExit: 1

/Users/divyasesh/Desktop/talmolab/sleap-core-docs/.venv/lib/python3.13/site-packages/IPython/core/interactiveshell.py:3707: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


As you can see, the loss and accuracy pick up from where it left off in the previous training.


Usually, however, if you're continuing training it's likely because you're starting off from an already trained model.

In this case, all you need to do to continue training is to create a new `Trainer` from the existing model configuration and provide the previous ckpt paths as below:

In [12]:
# Load config.
cfg = sleap.load_config("models/baseline_model.topdown")
# cfg.outputs.run_name = "new_folder"  # Set the run_name to a new value if you want the model to be saved to a different folder.

# Create and initialize the trainer.
trainer = sleap.nn.training.Trainer.from_config(cfg)
trainer.setup()

# Replace the randomly initialized weights with the saved weights.
trainer.keras_model.load_weights("models/baseline_model.topdown/best_model.h5")

INFO:sleap.nn.training:Loading training labels from: labels.pkg.slp
INFO:sleap.nn.training:Creating training and validation splits from validation fraction: 0.1
INFO:sleap.nn.training:  Splits: Training = 1440 / Validation = 160.
INFO:sleap.nn.training:Setting up for training...
INFO:sleap.nn.training:Setting up pipeline builders...
INFO:sleap.nn.training:Setting up model...
INFO:sleap.nn.training:Building test pipeline...
INFO:sleap.nn.training:Loaded test example. [0.925s]
INFO:sleap.nn.training:  Input shape: (160, 160, 1)
INFO:sleap.nn.training:Created Keras model.
INFO:sleap.nn.training:  Backbone: UNet(stacks=1, filters=16, filters_rate=2.0, kernel_size=3, stem_kernel_size=7, convs_per_block=2, stem_blocks=0, down_blocks=4, middle_block=True, up_blocks=2, up_interpolate=False, block_contraction=False)
INFO:sleap.nn.training:  Max stride: 16
INFO:sleap.nn.training:  Parameters: 2,101,501
INFO:sleap.nn.training:  Heads: 
INFO:sleap.nn.training:    [0] = CenteredInstanceConfmapsHead

In [13]:
trainer.config.optimization.epochs = 3
trainer.train()

INFO:sleap.nn.training:Creating tf.data.Datasets for training data generation...
INFO:sleap.nn.training:Finished creating training datasets. [17.7s]
INFO:sleap.nn.training:Starting training loop...
Epoch 1/3
360/360 - 9s - loss: 8.3664e-04 - head: 3.5190e-04 - thorax: 1.7037e-04 - abdomen: 9.8467e-04 - wingL: 7.9929e-04 - wingR: 8.0385e-04 - forelegL4: 0.0012 - forelegR4: 0.0012 - midlegL4: 9.5228e-04 - midlegR4: 9.8510e-04 - hindlegL4: 0.0013 - hindlegR4: 0.0013 - eyeL: 4.0772e-04 - eyeR: 3.9413e-04 - val_loss: 8.7351e-04 - val_head: 4.0943e-04 - val_thorax: 1.7453e-04 - val_abdomen: 9.4413e-04 - val_wingL: 8.3617e-04 - val_wingR: 8.4860e-04 - val_forelegL4: 0.0012 - val_forelegR4: 0.0012 - val_midlegL4: 9.4441e-04 - val_midlegR4: 0.0011 - val_hindlegL4: 0.0014 - val_hindlegR4: 0.0014 - val_eyeL: 4.4847e-04 - val_eyeR: 4.4179e-04 - lr: 1.0000e-04 - 9s/epoch - 24ms/step
Epoch 2/3
360/360 - 7s - loss: 8.0541e-04 - head: 3.4627e-04 - thorax: 1.6070e-04 - abdomen: 9.4325e-04 - wingL: 7.72

Output()

INFO:sleap.nn.evals:Saved predictions: models/baseline_model.topdown/labels_pr.train.slp
INFO:sleap.nn.evals:Saved metrics: models/baseline_model.topdown/metrics.train.npz
INFO:sleap.nn.evals:OKS mAP: 0.585451


Output()

INFO:sleap.nn.evals:Saved predictions: models/baseline_model.topdown/labels_pr.val.slp
INFO:sleap.nn.evals:Saved metrics: models/baseline_model.topdown/metrics.val.npz
INFO:sleap.nn.evals:OKS mAP: 0.574921


Again, the loss and accuracy pick up from where they left off prior to this round of training.

The resulting model can be used as usual for inference on new data.